In [ ]:
##################### Phase 3: Exploratory Data Analysis (EDA)& Clinical Analysis #####################

!pip install seaborn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Force the Cq_Value column to be strictly numeric before we plot anything
clean_df['Cq_Value'] = pd.to_numeric(clean_df['Cq_Value'], errors='coerce')

# 1. Calculate Positivity Counts per Channel
# Filter for rows where the Final_Call is '+'
positive_samples = clean_df[clean_df['Final_Call'] == '+']
positivity_counts = positive_samples.groupby('Target').size().reset_index(name='Positive_Count')

print("--- Positivity Counts per Channel ---")
display(positivity_counts)

# 2. Plotting Cq Value Distributions
# Drop NaN values so we are only plotting actual amplification numbers
positive_cq = clean_df.dropna(subset=['Cq_Value']).copy()

# Create the plot
plt.figure(figsize=(10, 6))
sns.boxplot(data=positive_cq, x='Target', y='Cq_Value', hue='Target', palette='viridis', legend=False)

# Add titles and labels
plt.title('Distribution of Cq Values per Optical Channel', fontsize=14, fontweight='bold')
plt.xlabel('Fluorescence Channel', fontsize=12)
plt.ylabel('Cq Value (Lower = Higher Concentration)', fontsize=12)

# Invert the Y-axis because in PCR, a lower Cq means more viral DNA
plt.gca().invert_yaxis() 

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# --- 1. Viral Load Density Plot ---
plt.figure(figsize=(10, 6))
# We use a KDE plot to see the smooth distribution of Cq values
sns.kdeplot(data=positive_cq, x='Cq_Value', hue='Target', fill=True, common_norm=False, palette='viridis', alpha=0.5)

plt.title('Density Distribution of Cq Values', fontsize=14, fontweight='bold')
plt.xlabel('Cq Value', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.gca().invert_xaxis() # Invert X so higher viral load is on the right
plt.tight_layout()
plt.show()

In [ ]:
# --- 3. 96-Well Plate Heatmap (CY5 Internal Control) ---
# We will check the CY5 channel, as it should amplify in every well
cy5_data = positive_cq[positive_cq['Target'] == 'CY5'].copy()

# Split the 'Well' column (e.g., 'A1') into Row ('A') and Column (1)
cy5_data['Row'] = cy5_data['Well'].str[0]
cy5_data['Col'] = cy5_data['Well'].str[1:].astype(int)

# Calculate the average Cq value for every specific well position across all 100 runs
plate_map = cy5_data.pivot_table(index='Row', columns='Col', values='Cq_Value', aggfunc='mean')

plt.figure(figsize=(12, 8))
# We plot the heatmap. If the edges are a drastically different color, the PCR machine needs maintenance!
sns.heatmap(plate_map, cmap='YlGnBu_r', annot=True, fmt=".1f", linewidths=.5)

plt.title('Spatial Heatmap: Average CY5 Cq Value by Well Position', fontsize=14, fontweight='bold')
plt.xlabel('Plate Column (1-12)', fontsize=12)
plt.ylabel('Plate Row (A-H)', fontsize=12)
plt.tight_layout()
plt.show()